In [ ]:
%gherkin
Feature: Masking invoice_number in the d_product_revenue table for customer privacy

  Background:
    Given the Unity Catalog is configured with purgo_playground schema
    And the purgo_playground.d_product_revenue table exists
    And the table d_product_revenue_clone does not exist

  Scenario: Clone the d_product_revenue table
    Given I am connected to the database
    When I drop the table d_product_revenue_clone if it exists
    Then I create a replica of the d_product_revenue table as d_product_revenue_clone

  Scenario Outline: Mask last 4 digits of invoice_number
    Given the d_product_revenue_clone table has been created
    When I apply masking logic to invoice_number column by replacing last 4 digits with "*"
    Then the invoice_number column should display <masked_invoice_number>

    Examples:
      | invoice_number | masked_invoice_number |
      | 1234234534     | 123423****             |
      | 9876543210     | 987654****             |

  Scenario: Validate masked invoice_number format
    Given the masked invoice_number column
    When I check the format of masked invoice_number
    Then the masked invoice_number should end with "****"

  Scenario: Error handling for non-existent table
    Given the attempt to clone a non-existent table named d_non_existing_table
    When I try to create a replica
    Then I should receive an error message "Table not found"

  Scenario: Error handling for masking invalid invoice_number
    Given an invoice_number of incorrect type "STRING"
    When I try masking the last 4 digits of invoice_number
    Then I should receive an error message "Invalid data type for invoice_number"

  Scenario Outline: Error handling for missing invoice_number
    Given an invoice with <invoice_number>
    When I perform masking operation
    Then I should receive a <error_message>

    Examples:
      | invoice_number | error_message                 |
      | NULL           | "Missing invoice_number"     |
      | ""             | "Empty invoice_number"       |
